# Notebook 03: Binder Comparison, Embedding Analysis & Integrated Ranking

```
[NB 01] Starling → AFRC filter → FoldMason → top conformers (_binder_ready/)
    ↓
RFDiffusion / BindCraft → binder structures + ipTM scores
Forge → binder sequences → ESMFold → structures
    ↓
[NB 03a] Binder contact filter → forge_binder_filter_passing.csv
                               → bindcraft_binder_filter_passing.csv
    ↓
[THIS NOTEBOOK]
  1. ESMFold remaining binder sequences (Forge)
  2. FoldMason on all binders → 3Di tokens + lDDT
  3. SaProt embeddings (sequence + 3Di) → UMAP → HDBSCAN clusters
  4. Contact maps → epitope heatmap (PHF6* / PHF6 / jR2R3)
  5. Chemical validation (GRAVY, charge, instability)
  6. Integrated score → experimental shortlist
    ↓
integrated_ranking.csv → experimental shortlist
```

**Requires**: GPU runtime (ESMFold, SaProt)  
**Runtime**: ~30–60 min depending on binder count

## 0. GPU check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch to GPU runtime (Runtime > Change runtime type > T4 GPU)')

In [ ]:
# ── Google Drive Setup ────────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/CHEM_280'
    _in_drive  = True
    print(f'Drive mounted. Base: {DRIVE_BASE}')
except Exception:
    DRIVE_BASE = '/content'
    _in_drive  = False
    print('Drive not available — looking for CSVs in /content/')

# Auto-copy CSVs from Drive to /content/ so notebook paths work unchanged
import glob as _glob, shutil as _shutil
if _in_drive:
    for _src in _glob.glob(os.path.join(DRIVE_BASE, 'results', 'nb02b_output', '*_passing.csv')):
        _dst = os.path.join('/content', os.path.basename(_src))
        _shutil.copy2(_src, _dst)
        print(f'Copied from Drive: {os.path.basename(_src)}')
    _forge_csv = os.path.join(DRIVE_BASE, 'results', 'nb02c_output', 'forge_binder_filter_passing.csv')
    if os.path.exists(_forge_csv):
        _shutil.copy2(_forge_csv, '/content/forge_binder_filter_passing.csv')
        print('Copied from Drive: forge_binder_filter_passing.csv')

NB03_OUT = os.path.join(DRIVE_BASE, 'results', 'nb03_output')
os.makedirs(NB03_OUT, exist_ok=True)
print(f'NB03 output dir: {NB03_OUT}')


## 1. Install dependencies

In [ ]:
%%capture
# Core bio + ML
!pip install transformers accelerate umap-learn hdbscan biopython MDAnalysis pandas matplotlib seaborn

# ESMFold (Meta's fold-from-sequence)
!pip install fair-esm

# FoldMason binary (structural MSA, provides 3Di tokens for SaProt)
!wget -q https://mmseqs.com/foldmason/foldmason-linux-avx2.tar.gz
!tar xzf foldmason-linux-avx2.tar.gz
!rm  foldmason-linux-avx2.tar.gz

In [ ]:
import os, json, glob, shutil, zipfile
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import MDAnalysis as mda

# Version checks
import importlib
for pkg in ['transformers', 'umap', 'hdbscan']:
    try:
        m = importlib.import_module(pkg)
        print(f'{pkg}: OK')
    except ImportError:
        print(f'{pkg}: MISSING')

r = subprocess.run(['./foldmason/bin/foldmason', 'version'], capture_output=True, text=True)
print('FoldMason:', r.stdout.strip()[:60])

## 2. Configuration

Edit this cell. All paths are relative to the Colab working directory.

In [ ]:
# ── INPUT PATHS ──────────────────────────────────────────────────────────────
# From Notebook 03a (binder contact filter)
FORGE_CSV      = 'forge_binder_filter_passing.csv'
BINDCRAFT_CSV  = 'bindcraft_binder_filter_passing.csv'

# Binder PDB directories (from 03a output or binder design tools)
FORGE_PDB_DIR      = 'results/esmfold/forge'       # ESMFolded Forge binders
BINDCRAFT_PDB_DIR  = 'results/esmfold/bindcraft'   # BindCraft output structures

# Known control binders (pulled from PDB — Tau antibodies, Baker 2025 miniproteins)
KNOWN_BINDERS_FASTA   = 'data/known_binders/sequences.fasta'
KNOWN_BINDERS_META    = 'data/known_binders/metadata.csv'
KNOWN_BINDERS_PDB_DIR = 'data/known_binders'       # pre-downloaded PDB structures

# Target conformer PDB (for contact map — best-ranking from NB01)
# Use a _binder_ready PDB from NB01 or the top-ranked conformer
TARGET_PDB = 'tau_k18_binder_ready/rank01_filtered_0001.pdb'

# ── EPITOPE RANGES (K18 1-indexed, matching NB01 config) ─────────────────────
EPITOPE_RANGES = [
    (2,  7,  'PHF6* VQIINK', 1.0),
    (33, 38, 'PHF6 VQIVYK',  1.0),
    (52, 70, 'jR2R3',         0.5),
]
EPITOPE_COLORS = {'PHF6* VQIINK': 'royalblue', 'PHF6 VQIVYK': 'crimson', 'jR2R3': 'forestgreen'}

# ── EMBEDDING MODEL ───────────────────────────────────────────────────────────
# 'saprot'  → SaProt 650M (sequence + 3Di joint embedding, recommended)
# 'esm2'    → ESM2 650M   (sequence only, fallback if SaProt too slow)
EMBEDDING_MODEL = 'saprot'

# ── SCORING WEIGHTS ───────────────────────────────────────────────────────────
SCORE_WEIGHTS = {
    'iptm':             0.30,   # interface pTM from BindCraft/RFDiffusion
    'plddt_interface':  0.20,   # ESMFold pLDDT at interface residues
    'cluster_conf':     0.15,   # HDBSCAN cluster membership probability
    'epitope_contact':  0.20,   # contacts PHF6* / PHF6 / jR2R3
    'chemical':         0.10,   # GRAVY + charge + instability score
    'known_proximity':  0.05,   # cosine similarity to AT8/HJ8.5/Baker2025
}

# ── THRESHOLDS ────────────────────────────────────────────────────────────────
CONTACT_DIST_A   = 8.0    # Å Cα–Cα contact threshold
TOP_N_FINAL      = 20     # candidates for experimental shortlist
UMAP_N_NEIGHBORS = 30
HDBSCAN_MIN_CLUST = 10

# ─────────────────────────────────────────────────────────────────────────────
os.makedirs('results/embeddings', exist_ok=True)
os.makedirs('results/contacts',   exist_ok=True)
os.makedirs('results/chemical',   exist_ok=True)
os.makedirs('results/foldmason/all_binders', exist_ok=True)
os.makedirs('tmp/foldmason_all_binders',     exist_ok=True)

print('Config loaded.')
print(f'Embedding model : {EMBEDDING_MODEL}')
print(f'Contact threshold : {CONTACT_DIST_A} Å')

## 3. Load binder inputs

In [ ]:
# ── Load passing binder tables from 03a ───────────────────────────────────────
dfs = []
for path, method in [(FORGE_CSV, 'Forge'), (BINDCRAFT_CSV, 'BindCraft')]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['method'] = method
        dfs.append(df)
        print(f'{method}: {len(df)} passing binders  ({path})')
    else:
        print(f'[MISSING] {path} — skipping {method}')

if not dfs:
    raise FileNotFoundError(
        'No binder CSVs found. Upload forge_binder_filter_passing.csv and/or '
        'bindcraft_binder_filter_passing.csv from Notebook 03a.')

binders = pd.concat(dfs, ignore_index=True)

# Required columns: 'name', 'sequence', 'iptm' (at minimum)
for col in ['name', 'sequence']:
    if col not in binders.columns:
        raise ValueError(f'Missing required column: {col}')
if 'iptm' not in binders.columns:
    print('WARNING: ipTM column not found — setting to 0.5 (unknown)')
    binders['iptm'] = 0.5

print(f'\nTotal binders loaded: {len(binders)}')
print(binders.groupby('method')['iptm'].describe().round(3))

# ── Load known controls ────────────────────────────────────────────────────────
if os.path.exists(KNOWN_BINDERS_META):
    known_meta = pd.read_csv(KNOWN_BINDERS_META)
    known_meta['method'] = 'known_control'
    known_meta['iptm']   = 1.0   # known binders = reference quality
    print(f'\nKnown controls: {len(known_meta)}')
    print(known_meta[['name', 'type', 'epitope']].to_string() if 'type' in known_meta.columns
          else known_meta['name'].tolist())
else:
    print(f'[MISSING] {KNOWN_BINDERS_META} — known controls not loaded')
    known_meta = pd.DataFrame(columns=['name', 'sequence', 'method', 'iptm'])

## 4. ESMFold binder sequences → structures

Folds any binder that does not yet have a PDB structure.
- Forge binders: sequence-only → fold here
- BindCraft binders: structures already exist → skip folding
- Known controls without a PDB structure → fold here

In [ ]:
import esm, torch

print('Loading ESMFold model (may take ~2 min)...')
esmfold = esm.pretrained.esmfold_v1()
esmfold = esmfold.eval().cuda()

def fold_sequence(seq, out_path):
    """Fold a single sequence with ESMFold. Saves PDB to out_path."""
    with torch.no_grad():
        output = esmfold.infer_pdb(seq)
    with open(out_path, 'w') as f:
        f.write(output)

print('ESMFold ready.')

In [ ]:
# Collect existing PDB paths; fold what's missing
# Expects: PDB file named {name}.pdb in the method's PDB directory
pdb_paths = {}

pdb_dir_map = {'Forge': FORGE_PDB_DIR, 'BindCraft': BINDCRAFT_PDB_DIR,
               'known_control': KNOWN_BINDERS_PDB_DIR}
os.makedirs(FORGE_PDB_DIR,     exist_ok=True)
os.makedirs(BINDCRAFT_PDB_DIR, exist_ok=True)

all_entries = pd.concat([binders, known_meta], ignore_index=True)
folded, skipped, missing = 0, 0, 0

for _, row in all_entries.iterrows():
    pdb_dir  = pdb_dir_map.get(row['method'], FORGE_PDB_DIR)
    pdb_path = os.path.join(pdb_dir, f"{row['name']}.pdb")

    if os.path.exists(pdb_path):
        pdb_paths[row['name']] = pdb_path
        skipped += 1
    elif pd.notna(row.get('sequence', None)) and len(str(row.get('sequence', ''))) > 10:
        seq = str(row['sequence']).upper().replace('*', '')
        fold_sequence(seq, pdb_path)
        pdb_paths[row['name']] = pdb_path
        folded += 1
        if folded % 10 == 0:
            print(f'  Folded {folded} structures...')
    else:
        print(f'  SKIP (no seq, no PDB): {row["name"]}')
        missing += 1

print(f'\nPDB structures: {len(pdb_paths)} total')
print(f'  Already existed: {skipped}  |  Newly folded: {folded}  |  Missing: {missing}')

In [ ]:
# Extract per-residue pLDDT (B-factor column in ESMFold PDB output)
def get_plddt(pdb_path):
    """Returns mean pLDDT of the structure from the B-factor column."""
    values = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM'):
                try:
                    values.append(float(line[60:66]))
                except ValueError:
                    pass
    return np.mean(values) if values else 0.0

all_entries['pdb_path'] = all_entries['name'].map(pdb_paths)
all_entries = all_entries[all_entries['pdb_path'].notna()].copy()
all_entries['plddt'] = all_entries['pdb_path'].apply(get_plddt)

print(f'Structures with pLDDT: {len(all_entries)}')
print(all_entries.groupby('method')['plddt'].describe().round(1))

## 5. FoldMason on all binders → 3Di tokens

Runs a structural MSA across all binder structures to:
1. Extract per-residue 3Di tokens (needed for SaProt embeddings)
2. Compute per-binder lDDT (structural quality relative to ensemble average)
3. Identify conserved structural motifs shared between Forge and BindCraft binders

In [ ]:
# Collect all binder PDBs into one directory for FoldMason
fm_input_dir = 'all_binders_for_foldmason'
os.makedirs(fm_input_dir, exist_ok=True)

for name, path in pdb_paths.items():
    dst = os.path.join(fm_input_dir, f'{name}.pdb')
    if not os.path.exists(dst):
        shutil.copy2(path, dst)

n_pdb = len(glob.glob(os.path.join(fm_input_dir, '*.pdb')))
print(f'Input PDBs for FoldMason: {n_pdb}')

# Run FoldMason
fm_prefix = 'results/foldmason/all_binders/msa'
fm_tmp    = 'tmp/foldmason_all_binders'
fm_cmd = [
    './foldmason/bin/foldmason', 'easy-msa',
    fm_input_dir, fm_prefix, fm_tmp,
    '--report-mode', '2',
    '--threads', '2',
]
print('Running FoldMason (structural MSA)...')
result = subprocess.run(fm_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('FoldMason failed')
print('Done.')
print('Files:', os.listdir('results/foldmason/all_binders/'))

In [ ]:
# Normalize JSON filename and parse 3Di tokens + lDDT per binder
fm_json_candidates = glob.glob('results/foldmason/all_binders/*.json')
if not fm_json_candidates:
    raise FileNotFoundError('No JSON in results/foldmason/all_binders/ — check FoldMason output')
fm_json = fm_json_candidates[0]
target  = 'results/foldmason/all_binders/foldmason.json'
if fm_json != target:
    shutil.copy2(fm_json, target)
    fm_json = target

with open(fm_json) as f:
    fm_data = json.load(f)

entries       = fm_data['entries']
fm_scores     = np.array(fm_data['scores'])
msa_lddt      = fm_data['statistics']['msaLDDT']

# Build name → 3Di string mapping
di3_map  = {}   # name → 3Di sequence string
lddt_map = {}   # name → lDDT score
aa_map   = {}   # name → amino acid sequence from FoldMason

for entry, score in zip(entries, fm_scores):
    stem = os.path.splitext(entry['name'])[0]
    di3_map[stem]  = entry['ss']
    lddt_map[stem] = score
    aa_map[stem]   = entry['aa']

print(f'FoldMason entries: {len(entries)}')
print(f'MSA lDDT: {msa_lddt:.4f}')
print(f'lDDT  mean={fm_scores.mean():.4f}  std={fm_scores.std():.4f}')

# Add lDDT to main table
all_entries['lddt_fm'] = all_entries['name'].map(lddt_map)

## 6. SaProt / ESM2 embeddings → UMAP → HDBSCAN

**SaProt** (`westlake-repl/SaProt_650M_AF2`): joint sequence+structure model trained on
interleaved `(amino_acid, 3Di_state)` token pairs. Encodes both modalities in one 1280-dim vector.

Input format: `'K v Q a I i I n K k ...'` (uppercase=aa, lowercase=3Di state per residue)

**ESM2 fallback** (`facebook/esm2_t33_650M_UR50D`): sequence-only, 1280-dim. Used if SaProt
download fails or is too slow on free Colab tier.

In [ ]:
from transformers import EsmTokenizer, EsmModel

MODEL_IDS = {
    'saprot': 'westlake-repl/SaProt_650M_AF2',
    'esm2':   'facebook/esm2_t33_650M_UR50D',
}

model_id = MODEL_IDS[EMBEDDING_MODEL]
print(f'Loading {model_id}...')

try:
    tokenizer = EsmTokenizer.from_pretrained(model_id)
    emb_model = EsmModel.from_pretrained(model_id).eval().cuda()
    print(f'Model loaded: {model_id}')
    using_saprot = (EMBEDDING_MODEL == 'saprot')
except Exception as e:
    print(f'Failed to load {model_id}: {e}')
    print('Falling back to ESM2...')
    fallback_id = MODEL_IDS['esm2']
    tokenizer   = EsmTokenizer.from_pretrained(fallback_id)
    emb_model   = EsmModel.from_pretrained(fallback_id).eval().cuda()
    using_saprot = False
    print('ESM2 loaded.')

print(f'\nUsing SaProt (joint seq+3Di): {using_saprot}')

In [ ]:
def build_saprot_input(aa_seq, di3_seq):
    """
    Build SaProt interleaved input: 'K v Q a I i ...'
    Pads 3Di to aa length with 'c' (coil) if shorter.
    """
    n = len(aa_seq)
    di3_padded = (di3_seq + 'c' * n)[:n].lower()
    tokens = ' '.join(a + d for a, d in zip(aa_seq.upper(), di3_padded))
    return tokens

def embed_batch(seqs, batch_size=8):
    """Mean-pool last hidden state → (N, 1280) embedding matrix."""
    all_embs = []
    for i in range(0, len(seqs), batch_size):
        batch  = seqs[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=512).to('cuda')
        with torch.no_grad():
            out = emb_model(**inputs)
        # Mean-pool over residue dimension (exclude [CLS] and [EOS])
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embs.append(emb.cpu().numpy())
        if (i // batch_size) % 5 == 0:
            print(f'  Embedded {min(i+batch_size, len(seqs))}/{len(seqs)}')
    return np.vstack(all_embs)

# Build input strings
input_seqs = []
valid_rows  = []

for _, row in all_entries.iterrows():
    name = row['name']
    aa   = aa_map.get(name, str(row.get('sequence', '')))
    if not aa:
        continue
    if using_saprot:
        di3 = di3_map.get(name, '')
        seq_input = build_saprot_input(aa, di3)
    else:
        seq_input = aa.upper()
    input_seqs.append(seq_input)
    valid_rows.append(row)

emb_df = pd.DataFrame(valid_rows).reset_index(drop=True)
print(f'Embedding {len(input_seqs)} binders...')
embeddings = embed_batch(input_seqs)
print(f'Embedding matrix: {embeddings.shape}')
np.save('results/embeddings/embeddings.npy', embeddings)
emb_df.to_csv('results/embeddings/embedding_metadata.csv', index=False)

In [ ]:
import umap
import hdbscan

# L2-normalize before UMAP
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_norm = embeddings / np.where(norms > 0, norms, 1)

reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
    n_jobs=1,
)
print('Running UMAP...')
umap_coords = reducer.fit_transform(emb_norm)
print(f'UMAP shape: {umap_coords.shape}')

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUST,
    metric='euclidean',
    prediction_data=True,
)
cluster_labels = clusterer.fit_predict(umap_coords)
cluster_probs  = hdbscan.all_points_membership_vectors(clusterer).max(axis=1)

emb_df['umap_x']       = umap_coords[:, 0]
emb_df['umap_y']       = umap_coords[:, 1]
emb_df['cluster']      = cluster_labels
emb_df['cluster_prob'] = cluster_probs

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_frac  = (cluster_labels == -1).mean()
print(f'Clusters: {n_clusters}  |  Noise: {noise_frac:.1%}')
emb_df.to_csv('results/embeddings/umap_clusters.csv', index=False)

In [ ]:
# UMAP plots — 4 colorings
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

method_palette = {
    'Forge':         '#2196F3',
    'BindCraft':     '#FF5722',
    'known_control': '#4CAF50',
}

def _scatter(ax, c, cmap=None, vmin=None, vmax=None, label_col=None, palette=None, title='', cbar_label=''):
    """Helper for consistent UMAP scatter plots."""
    noise_mask = emb_df['cluster'] == -1
    ax.scatter(emb_df.loc[noise_mask, 'umap_x'], emb_df.loc[noise_mask, 'umap_y'],
               c='lightgray', s=12, alpha=0.4, label='noise')
    if palette is not None:
        for grp, color in palette.items():
            m = emb_df[label_col] == grp
            ax.scatter(emb_df.loc[m, 'umap_x'], emb_df.loc[m, 'umap_y'],
                       c=color, s=25, alpha=0.8, label=grp)
        ax.legend(fontsize=8, markerscale=1.5)
    else:
        sc = ax.scatter(emb_df['umap_x'], emb_df['umap_y'],
                        c=c, cmap=cmap, vmin=vmin, vmax=vmax, s=20, alpha=0.8)
        plt.colorbar(sc, ax=ax, label=cbar_label, shrink=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# 1. By method
_scatter(axes[0,0], c=None, label_col='method', palette=method_palette,
         title='By generation method\n(overlap = convergent solutions)')

# 2. By ipTM
_scatter(axes[0,1], c=emb_df['iptm'], cmap='RdYlGn', vmin=0.5, vmax=1.0,
         title='By ipTM score\n(green = high confidence)', cbar_label='ipTM')

# 3. By HDBSCAN cluster
n_c = len(set(cluster_labels)) - 1
cmap_disc = plt.cm.get_cmap('tab20', n_c)
cluster_colors = [cmap_disc(c % n_c) if c >= 0 else (0.8, 0.8, 0.8, 0.4)
                  for c in cluster_labels]
axes[1,0].scatter(emb_df['umap_x'], emb_df['umap_y'],
                  c=cluster_colors, s=20, alpha=0.8)
axes[1,0].set_title(f'HDBSCAN clusters (N={n_c})\n(same cluster = same binding mode)',
                    fontsize=11)
axes[1,0].set_xlabel('UMAP 1'); axes[1,0].set_ylabel('UMAP 2')

# 4. By pLDDT
_scatter(axes[1,1], c=emb_df['plddt'], cmap='plasma', vmin=60, vmax=90,
         title='By ESMFold pLDDT\n(yellow = high confidence structure)', cbar_label='pLDDT')

# Annotate known controls on all panels
known_mask = emb_df['method'] == 'known_control'
for ax in axes.flat:
    for _, row in emb_df[known_mask].iterrows():
        ax.annotate(row['name'], (row['umap_x'], row['umap_y']),
                    fontsize=6, color='black',
                    xytext=(3, 3), textcoords='offset points')

plt.suptitle('Tau binder landscape — SaProt/ESM2 UMAP', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('results/embeddings/umap_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/embeddings/umap_plot.png')

## 7. Contact maps → epitope heatmap

For each binder with a **complex structure** (binder + target, two chains),
compute which target residues are within `CONTACT_DIST_A` Å (Cα–Cα) of any binder residue.

If complex structures are not available, loads pre-computed contact data from 03a
(`results/contacts/epitope_map.csv`).

In [ ]:
def compute_contacts_from_complex(pdb_path, binder_chain='B', target_chain='A',
                                  dist_thresh=CONTACT_DIST_A):
    """
    Returns a boolean array of length n_target_residues indicating which
    target residues are contacted by the binder.
    Chain A = target (K18), Chain B = binder.
    """
    u      = mda.Universe(pdb_path)
    target = u.select_atoms(f'chainID {target_chain} and name CA')
    binder = u.select_atoms(f'chainID {binder_chain} and name CA')
    if len(target) == 0 or len(binder) == 0:
        return None
    dists   = np.linalg.norm(
        target.positions[:, None, :] - binder.positions[None, :, :], axis=-1
    )
    return dists.min(axis=1) <= dist_thresh

# Try to compute contacts from complex PDBs in binder directories
# Falls back to loading pre-computed epitope_map.csv if no complex PDBs exist

epitope_precomputed = 'results/contacts/epitope_map.csv'
contact_matrix = {}   # name → bool array (target residues contacted)

for _, row in all_entries.iterrows():
    name     = row['name']
    pdb_dir  = pdb_dir_map.get(row['method'], FORGE_PDB_DIR)
    # BindCraft and RFDiffusion output complex PDBs
    complex_path = os.path.join(pdb_dir, f'{name}_complex.pdb')
    if os.path.exists(complex_path):
        contacts = compute_contacts_from_complex(complex_path)
        if contacts is not None:
            contact_matrix[name] = contacts

if contact_matrix:
    print(f'Contact maps from complex PDBs: {len(contact_matrix)}')
elif os.path.exists(epitope_precomputed):
    print(f'Loading pre-computed contact data from {epitope_precomputed}')
    epi_map_df = pd.read_csv(epitope_precomputed)
else:
    print('No complex PDBs and no pre-computed contact data found.')
    print('Epitope heatmap will be skipped.')
    print('→ To generate contact maps, ensure complex PDBs are named {name}_complex.pdb')
    epi_map_df = None

In [ ]:
if contact_matrix:
    # Build per-method contact frequency arrays
    n_target = len(next(iter(contact_matrix.values())))
    method_contacts = {}

    for method in ['Forge', 'BindCraft']:
        method_names = all_entries.loc[all_entries['method'] == method, 'name'].tolist()
        matrices     = [contact_matrix[n] for n in method_names if n in contact_matrix]
        if matrices:
            method_contacts[method] = np.stack(matrices).mean(axis=0)

    # Save epitope map CSV
    epi_map_df = pd.DataFrame({'residue': range(1, n_target+1)})
    for method, freq in method_contacts.items():
        epi_map_df[f'contact_freq_{method}'] = freq
    epi_map_df.to_csv(epitope_precomputed, index=False)

    # Plot
    fig, ax = plt.subplots(figsize=(16, 4))
    residues = np.arange(1, n_target+1)

    for method, color in [('Forge', '#2196F3'), ('BindCraft', '#FF5722')]:
        if method in method_contacts:
            ax.plot(residues, method_contacts[method], color=color,
                    lw=1.5, label=method, alpha=0.85)
            ax.fill_between(residues, method_contacts[method], alpha=0.15, color=color)

    for start, end, label, *_ in EPITOPE_RANGES:
        ax.axvspan(start, end, alpha=0.25, color=EPITOPE_COLORS.get(label, 'gray'), label=label)

    ax.set_xlabel('Target residue position (K18, 1-indexed)')
    ax.set_ylabel('Contact frequency')
    ax.set_title('Epitope contact frequency — Forge vs BindCraft\nShaded: PHF6* (blue), PHF6 (red), jR2R3 (green)')
    ax.legend(fontsize=9)
    ax.set_xlim(1, n_target)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('results/contacts/epitope_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved results/contacts/epitope_heatmap.png')

    # Per-binder epitope contact score
    def epitope_contact_score(name):
        if name not in contact_matrix:
            return 0.5   # unknown → neutral
        contacts = contact_matrix[name]
        fracs, weights = [], []
        for start, end, label, weight in EPITOPE_RANGES:
            idx = [i for i in range(start-1, end) if i < len(contacts)]
            f   = contacts[idx].mean() if idx else 0.0
            fracs.append(f); weights.append(weight)
        total_w = sum(weights)
        return sum(f*w for f, w in zip(fracs, weights)) / total_w

    all_entries['epitope_contact_score'] = all_entries['name'].apply(epitope_contact_score)
    print('Epitope contact scores computed.')
else:
    all_entries['epitope_contact_score'] = 0.5
    print('No contact maps — epitope_contact_score set to 0.5 (neutral).')

## 8. Chemical validation

Flags binders with properties that predict poor experimental behaviour:

| Property | Flag if |
|----------|---------|
| GRAVY score | > 0.5 (aggregation/solubility risk) |
| Net charge pH 7 | < −10 or > +10 |
| Isoelectric point | < 4 or > 10 |
| Helix+sheet fraction | < 20% (likely disordered binder) |
| Instability index | > 40 (unstable in solution) |

In [ ]:
def chemical_props(seq):
    """Returns dict of chemical properties via Biopython ProteinAnalysis."""
    # ProteinAnalysis requires standard 1-letter aa, no gaps
    clean = ''.join(c for c in str(seq).upper() if c.isalpha() and c != 'X')
    if len(clean) < 5:
        return {}
    try:
        pa      = ProteinAnalysis(clean)
        ss_frac = pa.secondary_structure_fraction()   # (helix, turn, sheet)
        return {
            'gravy':          round(pa.gravy(), 3),
            'pI':             round(pa.isoelectric_point(), 2),
            'charge_pH7':     round(pa.charge_at_pH(7.0), 2),
            'instability_idx': round(pa.instability_index(), 1),
            'helix_frac':     round(ss_frac[0], 3),
            'sheet_frac':     round(ss_frac[2], 3),
            'mw':             round(pa.molecular_weight(), 1),
        }
    except Exception:
        return {}

def chemical_score(row):
    """0–1 score: 1 = all flags pass, decreases with each flag."""
    props = row.get('_chem', {})
    if not props:
        return 0.5
    flags = [
        props.get('gravy', 0)          <= 0.5,
        abs(props.get('charge_pH7', 0)) <= 10,
        4 <= props.get('pI', 7)        <= 10,
        (props.get('helix_frac', 0) + props.get('sheet_frac', 0)) >= 0.2,
        props.get('instability_idx', 30) <= 40,
    ]
    return round(sum(flags) / len(flags), 3)

# Compute for all binders
seq_col = 'sequence' if 'sequence' in all_entries.columns else None
if seq_col:
    all_entries['_chem'] = all_entries[seq_col].apply(chemical_props)
    chem_df = pd.DataFrame(all_entries['_chem'].tolist())
    for col in chem_df.columns:
        all_entries[col] = chem_df[col].values
    all_entries['chemical_score'] = all_entries.apply(chemical_score, axis=1)
    all_entries.drop(columns=['_chem'], inplace=True, errors='ignore')
    chem_df['name']   = all_entries['name'].values
    chem_df['method'] = all_entries['method'].values
    chem_df.to_csv('results/chemical/properties.csv', index=False)
    print('Chemical properties saved.')
    print(all_entries.groupby('method')['chemical_score'].describe().round(3))

    # Flag summary
    n_flagged = (all_entries['chemical_score'] < 0.6).sum()
    print(f'\nBinders with chemical flags (score < 0.6): {n_flagged}/{len(all_entries)}')
else:
    print('No sequence column — skipping chemical validation.')
    all_entries['chemical_score'] = 0.5

## 9. Integrated binder score

```
Integrated Score = w_iptm              × ipTM
                 + w_plddt_interface   × ESMFold pLDDT (normalised 0–1)
                 + w_cluster_conf      × HDBSCAN cluster probability
                 + w_epitope_contact   × epitope contact score
                 + w_chemical          × chemical score
                 + w_known_proximity   × cosine similarity to known controls
```

All sub-scores are in [0, 1]. Weights are set in the config cell.

In [ ]:
# ── Known control proximity ────────────────────────────────────────────────────
# Cosine similarity of each binder to the centroid of known control embeddings
known_mask_emb = emb_df['method'] == 'known_control'
if known_mask_emb.any():
    known_indices  = emb_df.index[known_mask_emb].tolist()
    centroid       = emb_norm[known_indices].mean(axis=0)
    centroid      /= max(np.linalg.norm(centroid), 1e-8)
    known_sim      = (emb_norm @ centroid).clip(0, 1)   # cosine → [0,1]
    emb_df['known_proximity'] = known_sim
else:
    print('No known controls in embedding — known_proximity set to 0.5')
    emb_df['known_proximity'] = 0.5

# ── Merge embedding features back into all_entries ────────────────────────────
merge_cols = ['name', 'umap_x', 'umap_y', 'cluster', 'cluster_prob', 'known_proximity']
all_entries = all_entries.merge(emb_df[merge_cols], on='name', how='left')

# ── Normalise pLDDT to [0,1] ──────────────────────────────────────────────────
plddt_lo, plddt_hi = 50.0, 90.0
all_entries['plddt_norm'] = ((all_entries['plddt'].fillna(plddt_lo) - plddt_lo)
                             / (plddt_hi - plddt_lo)).clip(0, 1)

# ── Integrated score ─────────────────────────────────────────────────────────
W = SCORE_WEIGHTS
all_entries['integrated_score'] = (
    W['iptm']             * all_entries['iptm'].fillna(0.5)
  + W['plddt_interface']  * all_entries['plddt_norm']
  + W['cluster_conf']     * all_entries['cluster_prob'].fillna(0.0)
  + W['epitope_contact']  * all_entries['epitope_contact_score'].fillna(0.5)
  + W['chemical']         * all_entries['chemical_score'].fillna(0.5)
  + W['known_proximity']  * all_entries['known_proximity'].fillna(0.5)
).round(4)

ranking = (all_entries
           .sort_values('integrated_score', ascending=False)
           .reset_index(drop=True))
ranking.index += 1

show_cols = ['name', 'method', 'integrated_score', 'iptm', 'plddt', 'cluster',
             'epitope_contact_score', 'chemical_score']
show_cols = [c for c in show_cols if c in ranking.columns]
print('Top binders by integrated score:')
print(ranking[show_cols].head(TOP_N_FINAL).to_string())

ranking.to_csv('results/integrated_ranking.csv', index_label='rank')
print(f'\nFull ranking saved: results/integrated_ranking.csv')

In [ ]:
# Summary plots: score distribution + top-20 candidates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for method, color in method_palette.items():
    subset = ranking[ranking['method'] == method]
    ax.hist(subset['integrated_score'], bins=20, alpha=0.6, color=color,
            label=f'{method} (n={len(subset)})', edgecolor='white')
ax.set_xlabel('Integrated score')
ax.set_ylabel('Count')
ax.set_title('Integrated score distribution by method')
ax.legend(fontsize=9)

ax = axes[1]
top20 = ranking.head(TOP_N_FINAL)
colors_top = [method_palette.get(m, 'gray') for m in top20['method']]
ax.barh(range(len(top20)), top20['integrated_score'][::-1].values,
        color=colors_top[::-1], edgecolor='white')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['name'][::-1].values, fontsize=7)
ax.set_xlabel('Integrated score')
ax.set_title(f'Top {TOP_N_FINAL} candidates')
ax.axvline(0.7, color='black', ls='--', lw=1, alpha=0.5, label='0.7 threshold')
ax.legend(fontsize=8)

plt.suptitle('Integrated binder ranking', fontsize=13)
plt.tight_layout()
plt.savefig('results/integrated_ranking_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/integrated_ranking_plot.png')

## 10. Download all results

In [ ]:
try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

zip_name = 'tau_binder_comparison_results.zip'

collect = [
    ('results/integrated_ranking.csv',       'ranking.csv'),
    ('results/integrated_ranking_plot.png',  'ranking_plot.png'),
    ('results/embeddings/umap_plot.png',     'umap_plot.png'),
    ('results/embeddings/umap_clusters.csv', 'umap_clusters.csv'),
    ('results/contacts/epitope_heatmap.png', 'epitope_heatmap.png'),
    ('results/contacts/epitope_map.csv',     'epitope_map.csv'),
    ('results/chemical/properties.csv',      'chemical_properties.csv'),
    ('results/foldmason/all_binders/foldmason.json', 'foldmason_all_binders.json'),
]

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in collect:
        if os.path.exists(src):
            zf.write(src, arcname)
        else:
            print(f'  [SKIP] {src}')

print(f'Created: {zip_name}  ({os.path.getsize(zip_name)//1024} KB)')

if _in_drive:
    import shutil as _shutil2
    _shutil2.copy2(zip_name, os.path.join(NB03_OUT, os.path.basename(zip_name)))
    print(f'Saved to Drive: {NB03_OUT}')
if _in_colab:
    _colab_files.download(zip_name)
else:
    print(f'Not in Colab — file saved at: {os.path.abspath(zip_name)}')